In [2]:
#in_dir = "/Users/yerik/Music/_4_MUSIC_PROD/_0_PROD_MATERIAL/_ALL_3_PROD_stems_S/_25_07_DRUM/d_40_percent_silence"
in_dir  ="/Users/yerik/Music/_4_MUSIC_PROD/_0_PROD_MATERIAL/_ALL_3_PROD_stems_S"
out_dir = "/Users/yerik/Music/_0_YODJ_PROD/_MIDLIB_1_HIHAT"

# GET ONE HIGH HATPER SAMPLE 

In [3]:
# =========================================================
# -----######-----######  CORE IMPORTABLE FUNCTION  ######-----######-----######
# =========================================================

import os
import re
import shutil
import subprocess
import tempfile

import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from tqdm import tqdm


def _hat_0403_i1_GET_df_hatonly_mp3_from_folder(
    in_dir,
    out_dir,
    audio_extensions=(".mp3", ".MP3"),
    # audio
    sr_target=44100,
    hop_length=256,
    n_fft=2048,
    # filename BPM parse (used only as a safety cap)
    bpm_cap_beats=0.50,          # hats are usually <= 1/2 beat; bump to 1.0 if you want longer
    # start detection
    pre_ms=4,                    # tiny pre-roll
    min_gap_ms=60,               # hats can be fast; allow closer than kicks
    # hat specificity (FREQUENCY + TRANSIENT)
    hat_band=(6000, 16000),      # main hat energy
    low_band=(30, 250),          # reject low-end (kick/bass bleed)
    mid_band=(800, 3500),        # reject snares/claps if too dominant
    hat_over_low_ratio=8.0,      # must be VERY bright
    hat_over_mid_ratio=1.7,      # must be brighter than mid
    rms_gate_db=-50,
    peak_gate_db=-30,
    centroid_gate_hz=6500,       # spectral centroid must be high
    # end trimming (variable length = "just the hat")
    min_hat_ms=25,
    max_hat_ms=220,              # hard cap even if BPM is slow
    end_hold_ms=10,
    end_db_drop=18,              # end when hat-band drops this many dB from its local peak
    end_floor_db=-60,            # or below this level (relative to local peak ref)
    fade_ms=4,
    # mp3 export
    mp3_bitrate="320k",
    ffmpeg_path="ffmpeg",
    overwrite=True,
    max_files=None,
):
    """
    Recursively find all MP3 files.
    For each MP3:
      - Parse BPM from filename: last '-<BPM>.mp3' (used only for max duration cap)
      - Detect ONE best hi-hat transient:
          * strong hat-band (6k-16k) + high spectral centroid
          * reject low/mid dominance (avoid kicks/snares)
      - Trim tail using hat-band decay => "just the hat", variable length
      - Export as MP3 with SAME filename into out_dir
      - Originals untouched

    Returns:
      df_out: one row per mp3 with timings + export path + debug stats
    """

    os.makedirs(out_dir, exist_ok=True)

    # --- ensure ffmpeg exists ---
    if shutil.which(ffmpeg_path) is None:
        raise RuntimeError(
            f"ffmpeg not found on PATH as '{ffmpeg_path}'. "
            "Install via: brew install ffmpeg  (or pass ffmpeg_path)"
        )

    # --- gather files recursively ---
    exts = tuple(audio_extensions) if isinstance(audio_extensions, (list, tuple)) else (audio_extensions,)
    mp3_paths = []
    for root, _, files in os.walk(in_dir):
        for fn in files:
            if fn.startswith("._") or fn.startswith(".DS"):
                continue
            if fn.endswith(exts):
                mp3_paths.append(os.path.join(root, fn))

    mp3_paths = sorted(mp3_paths)
    if max_files is not None:
        mp3_paths = mp3_paths[: int(max_files)]

    # --- bpm parser: last "-<bpm>.mp3" ---
    re_bpm = re.compile(r"-([0-9]+(?:\.[0-9]+)?)\.mp3$", re.IGNORECASE)

    def _parse_bpm_from_name(path):
        base = os.path.basename(path)
        m = re_bpm.search(base)
        if not m:
            return None
        bpm = float(m.group(1))
        if bpm <= 0:
            return None
        return bpm

    def _peak_db(y_seg):
        pk = float(np.max(np.abs(y_seg)) + 1e-12)
        return float(20 * np.log10(pk))

    def _band_rms_db(S_mag, freqs, fr0, fr1, f0, f1):
        if fr1 <= fr0:
            return -120.0
        fmask = (freqs >= f0) & (freqs <= f1)
        band = S_mag[fmask, fr0:fr1]
        if band.size == 0:
            return -120.0
        val = float(np.sqrt(np.mean(band ** 2)) + 1e-12)
        return float(20 * np.log10(val))

    def _band_env_db(y_seg, sr, f0, f1):
        # STFT band energy envelope in dB, ref = max => peak at 0 dB
        S = np.abs(librosa.stft(y_seg, n_fft=n_fft, hop_length=hop_length)) + 1e-12
        freqs = librosa.fft_frequencies(sr=sr, n_fft=n_fft)
        fmask = (freqs >= f0) & (freqs <= f1)
        band_energy = np.sqrt(np.mean(S[fmask, :] ** 2, axis=0)) + 1e-12
        band_db = librosa.amplitude_to_db(band_energy, ref=np.max)
        return band_db

    rows = []

    for src_path in tqdm(mp3_paths, desc="TQM | mp3 → find 1 hi-hat → trim → export mp3", leave=True):
        base_fn = os.path.basename(src_path)
        out_path = os.path.join(out_dir, base_fn)

        try:
            bpm = _parse_bpm_from_name(src_path)
            # If BPM missing, we still work; we just fall back to max_hat_ms cap.
            beat_sec = (60.0 / bpm) if bpm else None
            bpm_cap_sec = (bpm_cap_beats * beat_sec) if beat_sec else None

            pre_s = pre_ms / 1000.0
            min_len_sec = min_hat_ms / 1000.0
            max_len_sec = max_hat_ms / 1000.0
            if bpm_cap_sec is not None:
                max_len_sec = min(max_len_sec, bpm_cap_sec)

            # load mono
            y, sr = librosa.load(src_path, sr=sr_target, mono=True)
            if y is None or len(y) < int(sr * 0.25):
                raise ValueError("Audio too short or unreadable.")

            # Onset candidates (transients)
            onset_env = librosa.onset.onset_strength(y=y, sr=sr, hop_length=hop_length)
            onset_frames = librosa.onset.onset_detect(
                onset_envelope=onset_env,
                sr=sr,
                hop_length=hop_length,
                backtrack=False,
                pre_max=4, post_max=4, pre_avg=4, post_avg=4,
                delta=0.15, wait=0
            )
            onset_times = librosa.frames_to_time(onset_frames, sr=sr, hop_length=hop_length)
            if len(onset_times) == 0:
                raise ValueError("No onsets detected.")

            # merge too-close hits
            min_gap_sec = min_gap_ms / 1000.0
            merged = []
            for t in onset_times:
                t = float(t)
                if not merged or (t - merged[-1]) >= min_gap_sec:
                    merged.append(t)

            # STFT mag (global) for fast band scoring
            S = np.abs(librosa.stft(y, n_fft=n_fft, hop_length=hop_length)) + 1e-12
            freqs = librosa.fft_frequencies(sr=sr, n_fft=n_fft)

            # RMS gate (relative to max)
            rms = librosa.feature.rms(y=y, frame_length=n_fft, hop_length=hop_length)[0]
            rms_db = librosa.amplitude_to_db(rms + 1e-12, ref=np.max)

            # Spectral centroid (frame-wise), used to confirm "bright"
            cent = librosa.feature.spectral_centroid(y=y, sr=sr, n_fft=n_fft, hop_length=hop_length)[0]

            best = None

            # Choose best hat START
            for t in merged:
                start_sec = max(0.0, t - pre_s)
                end_sec_tmp = min(len(y) / sr, start_sec + max_len_sec)

                a0 = int(start_sec * sr)
                a1 = int(end_sec_tmp * sr)
                if a1 <= a0 + int(min_len_sec * sr):
                    continue

                # local gates around onset
                fr = int((t * sr) / hop_length)
                fr0g = max(0, fr - 2)
                fr1g = min(len(rms_db), fr + 3)
                local_rms_db = float(np.max(rms_db[fr0g:fr1g]))

                # peak in first ~60ms
                pk_win = min(len(y), a0 + int(0.06 * sr))
                pk_db = _peak_db(y[a0:pk_win])

                if local_rms_db < rms_gate_db:
                    continue
                if pk_db < peak_gate_db:
                    continue

                # centroid gate (use max centroid near onset)
                c0 = max(0, fr - 2)
                c1 = min(len(cent), fr + 3)
                local_cent_hz = float(np.max(cent[c0:c1]))
                if local_cent_hz < centroid_gate_hz:
                    continue

                # band dominance in first ~90ms (hat is an early bright burst)
                short_end = min(len(y) / sr, start_sec + 0.09)
                a1s = int(short_end * sr)

                fr0b = max(0, int(a0 / hop_length))
                fr1b = min(S.shape[1], int(a1s / hop_length) + 1)

                hat_db = _band_rms_db(S, freqs, fr0b, fr1b, hat_band[0], hat_band[1])
                low_db = _band_rms_db(S, freqs, fr0b, fr1b, low_band[0], low_band[1])
                mid_db = _band_rms_db(S, freqs, fr0b, fr1b, mid_band[0], mid_band[1])

                hat_lin = 10 ** (hat_db / 20.0)
                low_lin = 10 ** (low_db / 20.0)
                mid_lin = 10 ** (mid_db / 20.0)

                ratio_hat_low = float(hat_lin / (low_lin + 1e-12))
                ratio_hat_mid = float(hat_lin / (mid_lin + 1e-12))

                if ratio_hat_low < hat_over_low_ratio:
                    continue
                if ratio_hat_mid < hat_over_mid_ratio:
                    continue

                # score: prioritize bright hat dominance + centroid + peak
                score = float((ratio_hat_low * 0.9) + (ratio_hat_mid * 0.9) + (local_cent_hz / 5000.0) + (pk_db / 10.0))

                if (best is None) or (score > best["score"]):
                    best = {
                        "onset_sec": t,
                        "start_sec": start_sec,
                        "score": score,
                        "ratio_hat_low": ratio_hat_low,
                        "ratio_hat_mid": ratio_hat_mid,
                        "local_cent_hz": local_cent_hz,
                        "peak_db_first60ms": pk_db,
                    }

            if best is None:
                raise ValueError("No hi-hat-like onset passed filters (try lowering ratios/gates).")

            # ---- trim END based on hat-band decay ----
            start_sec = best["start_sec"]
            start_samp = int(start_sec * sr)

            max_end_sec = min(len(y) / sr, start_sec + max_len_sec)
            max_end_samp = int(max_end_sec * sr)

            seg = y[start_samp:max_end_samp].copy()
            if len(seg) < int(min_len_sec * sr):
                raise ValueError("Segment too short after start selection.")

            hat_env_db = _band_env_db(seg, sr, hat_band[0], hat_band[1])  # 0 at peak

            hold_frames = max(1, int((end_hold_ms / 1000.0) * sr / hop_length))
            peak_frame = int(np.argmax(hat_env_db))

            drop_thr = -abs(end_db_drop)
            floor_thr = float(end_floor_db)

            # min end frame (don’t cut too short)
            min_end_samp = int(min_len_sec * sr)
            min_end_frame = max(0, int(min_end_samp / hop_length))

            end_frame = None
            for fr in range(max(min_end_frame, peak_frame + 1), len(hat_env_db) - hold_frames):
                window = hat_env_db[fr : fr + hold_frames]
                if np.all(window <= drop_thr) or np.all(window <= floor_thr):
                    end_frame = fr
                    break

            if end_frame is None:
                # fallback: use ~120ms typical hat length (or max cap if smaller)
                fallback_sec = min(max_len_sec, 0.12)
                end_samp_local = int(fallback_sec * sr)
            else:
                end_samp_local = int(end_frame * hop_length)

            end_samp_local = max(end_samp_local, int(min_len_sec * sr))
            end_samp_local = min(end_samp_local, len(seg))

            clip = seg[:end_samp_local].copy()

            # fade out to avoid clicks
            fade_len = int((fade_ms / 1000.0) * sr)
            if len(clip) > fade_len + 4:
                fade = np.linspace(1.0, 0.0, fade_len)
                clip[-fade_len:] *= fade

            # write temp wav then ffmpeg to mp3
            with tempfile.TemporaryDirectory() as td:
                tmp_wav = os.path.join(td, "tmp.wav")
                sf.write(tmp_wav, clip, sr)

                cmd = [
                    ffmpeg_path, "-y" if overwrite else "-n",
                    "-i", tmp_wav,
                    "-vn",
                    "-ar", str(sr),
                    "-ac", "1",
                    "-b:a", mp3_bitrate,
                    out_path
                ]
                p = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
                if p.returncode != 0:
                    raise RuntimeError(f"ffmpeg failed: {p.stderr[-600:]}")

            rows.append({
                "src_path": src_path,
                "src_file": base_fn,
                "bpm": bpm,
                "beat_sec": beat_sec,
                "onset_sec": best["onset_sec"],
                "start_sec": best["start_sec"],
                "dur_ms": float(len(clip) / sr * 1000.0),
                "score": best["score"],
                "ratio_hat_low": best["ratio_hat_low"],
                "ratio_hat_mid": best["ratio_hat_mid"],
                "centroid_hz": best["local_cent_hz"],
                "out_path": out_path,
                "error": None
            })

        except Exception as e:
            rows.append({
                "src_path": src_path,
                "src_file": base_fn,
                "bpm": None,
                "beat_sec": None,
                "onset_sec": None,
                "start_sec": None,
                "dur_ms": None,
                "score": None,
                "ratio_hat_low": None,
                "ratio_hat_mid": None,
                "centroid_hz": None,
                "out_path": None,
                "error": str(e)
            })

    return pd.DataFrame(rows)

In [4]:


audio_extensions = (".mp3", ".MP3")


df_hats = _hat_0403_i1_GET_df_hatonly_mp3_from_folder(
    in_dir=in_dir,
    out_dir=out_dir,
    audio_extensions=audio_extensions,
    sr_target=44100,
    # hat strictness (start here; this is already VERY hat-focused)
    hat_over_low_ratio=8.0,
    hat_over_mid_ratio=1.7,
    centroid_gate_hz=6500,
    # trimming
    end_db_drop=18,
    end_hold_ms=10,
    min_hat_ms=25,
    max_hat_ms=220,
    # bpm cap (safety)
    bpm_cap_beats=0.50,
    # mp3
    mp3_bitrate="320k",
    ffmpeg_path="ffmpeg",
    overwrite=True,
)

TQM | mp3 → find 1 hi-hat → trim → export mp3:  13%|██▏             | 10064/75549 [1:29:35<3:04:02,  5.93it/s]/Users/yerik/_apple_lib/_b_envs/2ms_env/lib/python3.11/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1454
  warnings.warn(
TQM | mp3 → find 1 hi-hat → trim → export mp3:  14%|██▏             | 10531/75549 [1:31:42<4:08:22,  4.36it/s]/Users/yerik/_apple_lib/_b_envs/2ms_env/lib/python3.11/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1701
  warnings.warn(
TQM | mp3 → find 1 hi-hat → trim → export mp3:  16%|██▌             | 12196/75549 [1:42:30<7:03:45,  2.49it/s]/Users/yerik/_apple_lib/_b_envs/2ms_env/lib/python3.11/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1593
  warnings.warn(
TQM | mp3 → find 1 hi-hat → trim → export mp3:  19%|███             | 14257/75549 [2:06:07<4:25:00,  3.85it/s]/User

# CUT the ones that are not one HAT


In [5]:
# ===========================================
# -----######-----######  CORE FUNCTION  ######
# ===========================================

import os
import shutil
import numpy as np
import pandas as pd
from tqdm import tqdm

def _hihat_0403_i2_GET_df_hihat_SCORE_mp3(
    folder_path,
    sr=44100,

    # --- duration / shape ---
    peak_must_be_within_ms=160,
    min_duration_ms=20,
    max_duration_ms=1400,

    # --- bands (wider + more forgiving) ---
    hat_band_hz=(4500, 18000),
    pres_band_hz=(2500, 9000),     # presence for many hats
    low_band_hz=(20, 300),
    mid_band_hz=(300, 2500),

    # --- ONE HIT ONLY gates (still important; slightly more forgiving) ---
    max_onset_peaks=1,
    onset_peak_wait_ms=120,
    onset_peak_rel_thresh=0.52,
    max_env_peaks=1,
    env_peak_rel_thresh=0.55,

    # --- envelope sanity ---
    max_postpeak_rise_frac=0.28,   # loosened
    retrigger_rms_rel=0.42,        # loosened
    retrigger_min_ms=120,

    # --- texture (loosened) ---
    min_spectral_flatness=0.14,    # loosened a lot
    min_zcr=0.045,                 # loosened a lot

    # --- scoring (KEY CHANGE) ---
    score_keep_threshold=3.2,      # lower => keeps more, higher => stricter
    score_weights=None,            # leave None to use defaults

    # --- file actions ---
    erase_mode="move",             # "move" (recommended) or "delete"
    quarantine_folder_name="_TRASH_NOT_HIHATS",
    dry_run=True,
    audio_extensions=(".mp3",),
    verbose=False,
):
    """
    Hi-hat detector using a weighted SCORE (more forgiving than hard cutoffs).
    Keeps samples likely to be a SINGLE hi-hat hit (closed or short-open), and
    removes non-hats immediately.

    Returns df with score + diagnostics.
    """

    import librosa

    if score_weights is None:
        # Balanced: frequency dominance matters most, but texture + shape help.
        score_weights = {
            "hat_vs_low": 1.35,
            "hat_vs_mid": 1.05,
            "presence":   0.70,
            "flatness":   0.55,
            "zcr":        0.55,
            "attack":     0.55,
            "decay":      0.45,
        }

    def _is_bad_hidden_file(fn):
        return fn.startswith("._") or fn.startswith(".DS") or fn.startswith("._DS")

    def _hz_to_bin(hz, n_fft, sr_):
        return int(np.clip(np.round(hz * n_fft / sr_), 0, n_fft // 2))

    def _count_peaks_simple(x, rel_thresh=0.6, min_dist=4):
        x = np.asarray(x, dtype=float)
        if len(x) < 5:
            return 0
        mx = float(np.max(x)) + 1e-12
        thr = rel_thresh * mx

        peaks = []
        for i in range(1, len(x) - 1):
            if x[i] > thr and x[i] >= x[i-1] and x[i] >= x[i+1]:
                if not peaks or (i - peaks[-1]) >= min_dist:
                    peaks.append(i)
        return len(peaks)

    def _clip01(v):
        return float(np.clip(v, 0.0, 1.0))

    def _score_log_ratio(r, lo, hi):
        """
        Map log10(r) into 0..1 based on lo..hi in log10 space.
        """
        r = float(max(r, 1e-12))
        x = np.log10(r)
        return _clip01((x - lo) / (hi - lo + 1e-12))

    def _analyze_one(path_mp3):
        out = {
            "Path": path_mp3,
            "file_name": os.path.basename(path_mp3),
            "decision": "UNKNOWN",
            "reason": "",
            "dur_ms": np.nan,
            "peak_time_ms": np.nan,

            "onset_peaks": np.nan,
            "env_peaks": np.nan,
            "postpeak_rise_frac": np.nan,
            "retrigger_flag": np.nan,

            "flatness": np.nan,
            "zcr": np.nan,

            "hat_low_ratio": np.nan,
            "hat_mid_ratio": np.nan,
            "presence_ratio": np.nan,

            "score": np.nan,
            "error": "",
        }

        try:
            y, sr_ = librosa.load(path_mp3, sr=sr, mono=True)
            if y is None or len(y) < 32:
                out["decision"] = "NOT_HIHAT"
                out["reason"] = "empty_audio"
                return out

            dur_ms = (len(y) / sr_) * 1000.0
            out["dur_ms"] = float(dur_ms)

            if dur_ms < min_duration_ms:
                out["decision"] = "NOT_HIHAT"
                out["reason"] = "too_short"
                return out
            if dur_ms > max_duration_ms:
                out["decision"] = "NOT_HIHAT"
                out["reason"] = "too_long"
                return out

            # normalize
            peak_abs = float(np.max(np.abs(y))) + 1e-12
            y = y / peak_abs

            # peak early
            pk_i = int(np.argmax(np.abs(y)))
            pk_t_ms = (pk_i / sr_) * 1000.0
            out["peak_time_ms"] = float(pk_t_ms)
            if pk_t_ms > peak_must_be_within_ms:
                out["decision"] = "NOT_HIHAT"
                out["reason"] = "peak_too_late"
                return out

            # start at peak
            y_post = y[pk_i:]
            if len(y_post) < 1024:
                out["decision"] = "NOT_HIHAT"
                out["reason"] = "too_short_postpeak"
                return out

            hop = 256
            frame_len = 1024

            # ---- ONE HIT gates (still hard, but looser thresholds) ----
            onset_env = librosa.onset.onset_strength(y=y_post, sr=sr_, hop_length=hop)
            if onset_env is None or len(onset_env) < 6:
                out["decision"] = "NOT_HIHAT"
                out["reason"] = "onset_failed"
                return out

            min_dist_frames = max(1, int((onset_peak_wait_ms / 1000.0) * sr_ / hop))
            onset_peaks = _count_peaks_simple(onset_env, rel_thresh=onset_peak_rel_thresh, min_dist=min_dist_frames)
            out["onset_peaks"] = int(onset_peaks)
            if onset_peaks > max_onset_peaks:
                out["decision"] = "NOT_HIHAT"
                out["reason"] = "multi_hit_onsets"
                return out

            rms = librosa.feature.rms(y=y_post, frame_length=frame_len, hop_length=hop, center=False)[0]
            if rms is None or len(rms) < 8:
                out["decision"] = "NOT_HIHAT"
                out["reason"] = "rms_failed"
                return out

            rms = np.maximum(rms, 1e-9)
            rms = rms / (np.max(rms) + 1e-12)

            env_peaks = _count_peaks_simple(rms, rel_thresh=env_peak_rel_thresh, min_dist=min_dist_frames)
            out["env_peaks"] = int(env_peaks)
            if env_peaks > max_env_peaks:
                out["decision"] = "NOT_HIHAT"
                out["reason"] = "multi_peak_envelope"
                return out

            diffs = np.diff(rms)
            rises = np.sum(diffs > 0)
            out["postpeak_rise_frac"] = float(rises / max(len(diffs), 1))
            if out["postpeak_rise_frac"] > max_postpeak_rise_frac:
                out["decision"] = "NOT_HIHAT"
                out["reason"] = "envelope_rises_too_much"
                return out

            # retrigger check
            t_ms = (np.arange(len(rms)) * hop / sr_) * 1000.0
            start_idx = int(np.argmax(t_ms >= retrigger_min_ms)) if np.any(t_ms >= retrigger_min_ms) else len(rms)
            retrigger_flag = 0
            if start_idx < len(rms):
                tail = rms[start_idx:]
                if len(tail) >= 4:
                    min_tail = float(np.min(tail))
                    max_tail = float(np.max(tail))
                    if (max_tail - min_tail) > 0.50 and max_tail > retrigger_rms_rel:
                        retrigger_flag = 1
            out["retrigger_flag"] = int(retrigger_flag)
            if retrigger_flag == 1:
                out["decision"] = "NOT_HIHAT"
                out["reason"] = "retrigger_detected"
                return out

            # ---- texture (SOFT threshold, but still reject super-tonal stuff) ----
            flat = librosa.feature.spectral_flatness(y=y_post)[0]
            flat_v = float(np.median(flat)) if flat is not None and len(flat) else 0.0
            out["flatness"] = float(flat_v)
            if flat_v < min_spectral_flatness:
                out["decision"] = "NOT_HIHAT"
                out["reason"] = "too_tonal_not_noisy"
                return out

            zcr = librosa.feature.zero_crossing_rate(y_post, frame_length=1024, hop_length=hop, center=False)[0]
            zcr_v = float(np.median(zcr)) if zcr is not None and len(zcr) else 0.0
            out["zcr"] = float(zcr_v)
            if zcr_v < min_zcr:
                out["decision"] = "NOT_HIHAT"
                out["reason"] = "zcr_too_low"
                return out

            # ---- spectral energy ratios ----
            cap = min(len(y_post), int(0.7 * sr_))
            y_cap = y_post[:cap] if cap > 512 else y_post

            n_fft = 4096
            S = np.abs(librosa.stft(y_cap, n_fft=n_fft, hop_length=hop, center=False)) ** 2
            if S is None or S.size == 0:
                out["decision"] = "NOT_HIHAT"
                out["reason"] = "stft_failed"
                return out

            hb0 = _hz_to_bin(hat_band_hz[0], n_fft, sr_)
            hb1 = _hz_to_bin(hat_band_hz[1], n_fft, sr_)
            pb0 = _hz_to_bin(pres_band_hz[0], n_fft, sr_)
            pb1 = _hz_to_bin(pres_band_hz[1], n_fft, sr_)
            lb0 = _hz_to_bin(low_band_hz[0], n_fft, sr_)
            lb1 = _hz_to_bin(low_band_hz[1], n_fft, sr_)
            mb0 = _hz_to_bin(mid_band_hz[0], n_fft, sr_)
            mb1 = _hz_to_bin(mid_band_hz[1], n_fft, sr_)

            e_hat = float(np.sum(S[hb0:hb1 + 1, :])) + 1e-12
            e_pres = float(np.sum(S[pb0:pb1 + 1, :])) + 1e-12
            e_low = float(np.sum(S[lb0:lb1 + 1, :])) + 1e-12
            e_mid = float(np.sum(S[mb0:mb1 + 1, :])) + 1e-12

            hat_low_ratio = e_hat / e_low
            hat_mid_ratio = e_hat / e_mid
            presence_ratio = e_pres / e_mid

            out["hat_low_ratio"] = float(hat_low_ratio)
            out["hat_mid_ratio"] = float(hat_mid_ratio)
            out["presence_ratio"] = float(presence_ratio)

            # ---- SCORE (0..~7ish) ----
            # ratios scored in log-space so “reasonable hats” still score.
            s_hat_low = _score_log_ratio(hat_low_ratio, lo=0.55, hi=1.35)     # ~3.5x..22x
            s_hat_mid = _score_log_ratio(hat_mid_ratio, lo=0.20, hi=0.95)     # ~1.6x..9x
            s_pres    = _score_log_ratio(presence_ratio, lo=-0.05, hi=0.55)   # ~0.9x..3.5x

            s_flat = _clip01((flat_v - 0.12) / (0.38 - 0.12 + 1e-12))
            s_zcr  = _clip01((zcr_v - 0.04) / (0.18 - 0.04 + 1e-12))

            # attack: peak early gets higher score
            s_attack = _clip01((peak_must_be_within_ms - pk_t_ms) / peak_must_be_within_ms)

            # decay: fewer rises is better
            s_decay = _clip01((0.35 - out["postpeak_rise_frac"]) / 0.35)

            score = (
                score_weights["hat_vs_low"] * s_hat_low +
                score_weights["hat_vs_mid"] * s_hat_mid +
                score_weights["presence"]   * s_pres +
                score_weights["flatness"]   * s_flat +
                score_weights["zcr"]        * s_zcr +
                score_weights["attack"]     * s_attack +
                score_weights["decay"]      * s_decay
            )
            out["score"] = float(score)

            if score >= score_keep_threshold:
                out["decision"] = "HIHAT"
                out["reason"] = "passed_SCORE"
            else:
                out["decision"] = "NOT_HIHAT"
                out["reason"] = "score_too_low"

            return out

        except Exception as e:
            out["decision"] = "NOT_HIHAT"
            out["reason"] = "exception"
            out["error"] = str(e)
            return out

    folder_path = os.path.abspath(os.path.expanduser(str(folder_path)))
    if not os.path.isdir(folder_path):
        raise ValueError(f"folder_path not found: {folder_path}")

    exts_lower = tuple([e.lower() for e in audio_extensions])

    files = []
    for fn in os.listdir(folder_path):
        if _is_bad_hidden_file(fn):
            continue
        full = os.path.join(folder_path, fn)
        if os.path.isfile(full) and fn.lower().endswith(exts_lower):
            files.append(full)

    quarantine_dir = os.path.join(folder_path, quarantine_folder_name)
    if erase_mode == "move" and (not dry_run):
        os.makedirs(quarantine_dir, exist_ok=True)

    rows = []
    for p in tqdm(files, desc="HIHAT SCORE — scanning mp3"):
        res = _analyze_one(p)
        rows.append(res)

        if res["decision"] == "NOT_HIHAT" and (not dry_run):
            if erase_mode == "move":
                dest = os.path.join(quarantine_dir, os.path.basename(p))
                if os.path.exists(dest):
                    base, ext = os.path.splitext(os.path.basename(p))
                    k = 1
                    while True:
                        dest2 = os.path.join(quarantine_dir, f"{base}__dup{k}{ext}")
                        if not os.path.exists(dest2):
                            dest = dest2
                            break
                        k += 1
                shutil.move(p, dest)
            elif erase_mode == "delete":
                os.remove(p)

    df = pd.DataFrame(rows)
    df["is_hihat"] = df["decision"].eq("HIHAT")
    df["action"] = "KEEP"
    df.loc[df["decision"].eq("NOT_HIHAT"), "action"] = ("DRY_RUN_SKIP" if dry_run else ("MOVE_TO_QUARANTINE" if erase_mode=="move" else "DELETE"))

    if verbose:
        n_all = len(df)
        n_h = int(df["is_hihat"].sum())
        n_nh = n_all - n_h
        print("\n---- HIHAT SCORE SUMMARY ----")
        print(f"folder: {folder_path}")
        print(f"mp3 scanned: {n_all}")
        print(f"KEEP (HIHAT): {n_h}")
        print(f"ERASE (NOT_HIHAT): {n_nh}")
        print(f"dry_run: {dry_run} | erase_mode: {erase_mode}")
        print(f"score_keep_threshold: {score_keep_threshold}")
        if erase_mode == "move":
            print(f"quarantine_dir: {quarantine_dir}")

    return df

In [6]:
folder_path = out_dir

# 1) Preview
df_hats = _hihat_0403_i2_GET_df_hihat_SCORE_mp3(
    folder_path=folder_path,
    dry_run=True,
    erase_mode="move",
    score_keep_threshold=3.2,   # <<< keeps MORE hats (try 3.0 if still too tight)
    verbose=True
)

# 2) Execute
df_hats = _hihat_0403_i2_GET_df_hihat_SCORE_mp3(
    folder_path=folder_path,
    dry_run=False,
    erase_mode="move",
    score_keep_threshold=3.2,
    verbose=True
)

HIHAT SCORE — scanning mp3:   0%|                                                    | 0/7996 [00:00<?, ?it/s]/Users/yerik/_apple_lib/_b_envs/2ms_env/lib/python3.11/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1991
  warnings.warn(
/Users/yerik/_apple_lib/_b_envs/2ms_env/lib/python3.11/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1380
  warnings.warn(
/Users/yerik/_apple_lib/_b_envs/2ms_env/lib/python3.11/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1070
  warnings.warn(
/Users/yerik/_apple_lib/_b_envs/2ms_env/lib/python3.11/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1672
  warnings.warn(
/Users/yerik/_apple_lib/_b_envs/2ms_env/lib/python3.11/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input sign


---- HIHAT SCORE SUMMARY ----
folder: /Users/yerik/Music/_0_YODJ_PROD/_MIDLIB_1_HIHAT
mp3 scanned: 7996
KEEP (HIHAT): 0
ERASE (NOT_HIHAT): 7996
dry_run: True | erase_mode: move
score_keep_threshold: 3.2
quarantine_dir: /Users/yerik/Music/_0_YODJ_PROD/_MIDLIB_1_HIHAT/_TRASH_NOT_HIHATS


HIHAT SCORE — scanning mp3:   0%|                                            | 5/7996 [00:00<07:12, 18.46it/s]/Users/yerik/_apple_lib/_b_envs/2ms_env/lib/python3.11/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1991
  warnings.warn(
/Users/yerik/_apple_lib/_b_envs/2ms_env/lib/python3.11/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1380
  warnings.warn(
HIHAT SCORE — scanning mp3:   0%|                                           | 16/7996 [00:01<10:17, 12.91it/s]/Users/yerik/_apple_lib/_b_envs/2ms_env/lib/python3.11/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1070
  warnings.warn(
HIHAT SCORE — scanning mp3:   0%|▏                                          | 24/7996 [00:01<08:14, 16.13it/s]/Users/yerik/_apple_lib/_b_envs/2ms_env/lib/python3.11/site-packages/librosa/core/spectrum.py:266: UserWarning: n_f


---- HIHAT SCORE SUMMARY ----
folder: /Users/yerik/Music/_0_YODJ_PROD/_MIDLIB_1_HIHAT
mp3 scanned: 7996
KEEP (HIHAT): 0
ERASE (NOT_HIHAT): 7996
dry_run: False | erase_mode: move
score_keep_threshold: 3.2
quarantine_dir: /Users/yerik/Music/_0_YODJ_PROD/_MIDLIB_1_HIHAT/_TRASH_NOT_HIHATS


# organize KICKS in folders 

In [7]:
# ============================================================
# 0_FNS
# ============================================================

import os
import shutil
import numpy as np
import pandas as pd
from tqdm import tqdm
import librosa


# -----######-----######  CORE IMPORTABLE FUNCTION  #####-----######-----######
def _hat_0403_smartbucket_inplace_GET_df_manifest(
    in_dir,
    audio_extensions,
    mode="move",                  # "move" | "copy" | "none"
    sr_target=44100,
    top_db_trim=70,               # hats often have quiet tails; don't trim too aggressively
    n_fft=4096,                   # better HF resolution
    hop_length=256,
    min_per_bucket=10,            # prevents empty folders
    dry_run=False,
    seed=7,
):
    """
    Smart hi-hat organizer (in-place):
    - Reads audio files from in_dir (root only)
    - Creates hat bucket folders inside in_dir
    - Assigns with hat-specific multi-check scoring (HF bands are key)
    - Rebalances borderline files to avoid empty folders
    - Moves (or copies) originals into bucket folders
    - Saves hat_manifest.csv into in_dir
    Returns: df_manifest, df_summary
    """

    rng = np.random.RandomState(seed)

    # --- Hat folders (production-first) ---
    # (You can rename these later, but these buckets are actually useful in practice.)
    buckets = [
        "01_CLOSED_TIGHT",         # short, clean tick
        "02_CLOSED_TICKY",         # clicky / bright / crisp
        "03_OPEN_WASHY",           # long decay, airy
        "04_OPEN_METALLIC",        # long + tonal peaks / metallic ring
        "05_SHAKER_HAT",           # shaker-like noise band
        "06_NOISY_TEXTURE",        # noisy / flat / gritty
        "07_CRASHY_HAT",           # very long, wideband (almost cymbal-ish)
        "08_LOFI_DIRTY",           # dirt + rolled highs / crunchy
        "09_WEIRD_FX",             # leftover catcher for oddities
    ]
    bucket_paths = {b: os.path.join(in_dir, b) for b in buckets}

    # ---------------- helpers ----------------
    def _safe_makedirs(p):
        os.makedirs(p, exist_ok=True)

    def _is_in_bucket_folder(path_abs):
        for b in buckets:
            b_abs = os.path.abspath(bucket_paths[b])
            if os.path.abspath(path_abs).startswith(b_abs + os.sep):
                return True
        return False

    def _list_audio_files(root, exts):
        exts_l = [e.lower() for e in exts] if exts else []
        paths = []
        for fn in os.listdir(root):
            if fn.startswith("._") or fn.startswith(".DS"):
                continue
            p = os.path.join(root, fn)
            if os.path.isfile(p):
                if _is_in_bucket_folder(p):
                    continue
                ext = os.path.splitext(fn)[1].lower()
                if (not exts_l) or (ext in exts_l):
                    paths.append(p)
        return sorted(paths)

    def _mk_dest(dst_dir, src):
        dst = os.path.join(dst_dir, os.path.basename(src))
        if not os.path.exists(dst):
            return dst
        base, ext = os.path.splitext(os.path.basename(src))
        i = 1
        while True:
            dst2 = os.path.join(dst_dir, f"{base}__DUP{i}{ext}")
            if not os.path.exists(dst2):
                return dst2
            i += 1

    def _band_energy_ratio(S, freqs, f_lo, f_hi):
        mask = (freqs >= f_lo) & (freqs < f_hi)
        if not np.any(mask):
            return 0.0
        num = float(np.sum(S[mask, :]))
        den = float(np.sum(S)) + 1e-12
        return num / den

    def _decay_ms_from_peak(x, sr, drop_db):
        env = np.abs(x)
        if env.size < 10:
            return 0.0
        win = max(16, int(0.004 * sr))  # ~4ms smoothing
        k = np.ones(win) / win
        env_s = np.convolve(env, k, mode="same")

        peak_idx = int(np.argmax(env_s))
        peak_val = float(env_s[peak_idx]) + 1e-12
        target = peak_val * (10 ** (-drop_db / 20.0))

        tail = env_s[peak_idx:]
        below = np.where(tail <= target)[0]
        if below.size == 0:
            return (len(tail) / sr) * 1000.0
        return (float(below[0]) / sr) * 1000.0

    def _q(series, p):
        return float(np.nanpercentile(series.to_numpy(dtype=float), p))

    def _z_factory(df_ok, col):
        v = df_ok[col].to_numpy(dtype=float)
        mu = float(np.nanmean(v))
        sd = float(np.nanstd(v) + 1e-12)
        def _z(vv):
            return (float(vv) - mu) / sd if sd > 1e-12 else 0.0
        return _z

    # ---------------- setup folders ----------------
    for b in buckets:
        _safe_makedirs(bucket_paths[b])

    paths = _list_audio_files(in_dir, audio_extensions)

    # ---------------- feature extraction ----------------
    rows = []
    for p in tqdm(paths, desc="Extracting hi-hat features", total=len(paths)):
        try:
            y, sr = librosa.load(p, sr=sr_target, mono=True)
            y, _ = librosa.effects.trim(y, top_db=top_db_trim)
            if y.size == 0:
                raise ValueError("empty_audio_after_trim")

            y = y / (np.max(np.abs(y)) + 1e-12)

            # STFT magnitude
            S = np.abs(librosa.stft(y, n_fft=n_fft, hop_length=hop_length)) + 1e-12
            freqs = librosa.fft_frequencies(sr=sr, n_fft=n_fft)

            # ---- THE IMPORTANT BANDS FOR HATS ----
            low_ratio   = _band_energy_ratio(S, freqs, 20, 300)         # hats should be low
            mid_ratio   = _band_energy_ratio(S, freqs, 300, 2000)
            pres_ratio  = _band_energy_ratio(S, freqs, 2000, 6000)      # presence
            hf_ratio    = _band_energy_ratio(S, freqs, 6000, 16000)     # hi-hat "air/metal"
            air_ratio   = _band_energy_ratio(S, freqs, 10000, 20000)    # air band (if sr supports)

            centroid = float(np.mean(librosa.feature.spectral_centroid(S=S, sr=sr)))
            rolloff  = float(np.mean(librosa.feature.spectral_rolloff(S=S, sr=sr, roll_percent=0.90)))
            flatness = float(np.mean(librosa.feature.spectral_flatness(S=S)))
            zcr      = float(np.mean(librosa.feature.zero_crossing_rate(y)))

            # decay + duration
            decay_ms_24 = _decay_ms_from_peak(y, sr, drop_db=24)        # hats: longer tails matter
            decay_ms_12 = _decay_ms_from_peak(y, sr, drop_db=12)        # early decay
            dur_ms      = (len(y) / sr) * 1000.0

            # transient sharpness proxy (first 30ms vs whole)
            early = y[: min(len(y), int(0.03 * sr))]
            early_rms = float(np.sqrt(np.mean(early**2) + 1e-12))
            full_rms  = float(np.sqrt(np.mean(y**2) + 1e-12))
            sharp = float(early_rms / (full_rms + 1e-12))

            # tonal/metallic indicator:
            # metallic hats often show clearer peaks => lower flatness + higher centroid/rolloff
            metallic_hint = float((1.0 - flatness) * (centroid / (rolloff + 1e-9)))

            rows.append({
                "Path": p,
                "file_name": os.path.basename(p),
                "dur_ms": dur_ms,
                "decay12_ms": decay_ms_12,
                "decay24_ms": decay_ms_24,
                "low_ratio": low_ratio,
                "mid_ratio": mid_ratio,
                "pres_ratio": pres_ratio,
                "hf_ratio": hf_ratio,
                "air_ratio": air_ratio,
                "centroid_hz": centroid,
                "rolloff_hz": rolloff,
                "flatness": flatness,
                "zcr": zcr,
                "sharp": sharp,
                "metallic_hint": metallic_hint,
                "error": ""
            })

        except Exception as e:
            rows.append({
                "Path": p,
                "file_name": os.path.basename(p),
                "dur_ms": np.nan,
                "decay12_ms": np.nan,
                "decay24_ms": np.nan,
                "low_ratio": np.nan,
                "mid_ratio": np.nan,
                "pres_ratio": np.nan,
                "hf_ratio": np.nan,
                "air_ratio": np.nan,
                "centroid_hz": np.nan,
                "rolloff_hz": np.nan,
                "flatness": np.nan,
                "zcr": np.nan,
                "sharp": np.nan,
                "metallic_hint": np.nan,
                "error": str(e)
            })

    df = pd.DataFrame(rows)
    df_ok = df[df["error"].eq("")].copy()

    if len(df_ok) == 0:
        df.to_csv(os.path.join(in_dir, "hat_manifest.csv"), index=False)
        summary = pd.DataFrame({"bucket": buckets, "count": [0]*len(buckets)})
        return df, summary

    # ---------------- adaptive percentiles ----------------
    Q = {
        "dur_lo":    _q(df_ok["dur_ms"], 25),
        "dur_hi":    _q(df_ok["dur_ms"], 75),
        "d24_lo":    _q(df_ok["decay24_ms"], 25),
        "d24_hi":    _q(df_ok["decay24_ms"], 75),
        "hf_hi":     _q(df_ok["hf_ratio"], 75),
        "hf_lo":     _q(df_ok["hf_ratio"], 25),
        "air_hi":    _q(df_ok["air_ratio"], 75),
        "flat_hi":   _q(df_ok["flatness"], 75),
        "flat_lo":   _q(df_ok["flatness"], 25),
        "low_hi":    _q(df_ok["low_ratio"], 75),
        "sharp_hi":  _q(df_ok["sharp"], 75),
        "cent_hi":   _q(df_ok["centroid_hz"], 75),
        "roll_hi":   _q(df_ok["rolloff_hz"], 75),
    }

    # z-score factories
    z_dur   = _z_factory(df_ok, "dur_ms")
    z_d24   = _z_factory(df_ok, "decay24_ms")
    z_hf    = _z_factory(df_ok, "hf_ratio")
    z_air   = _z_factory(df_ok, "air_ratio")
    z_low   = _z_factory(df_ok, "low_ratio")
    z_flat  = _z_factory(df_ok, "flatness")
    z_zcr   = _z_factory(df_ok, "zcr")
    z_sharp = _z_factory(df_ok, "sharp")
    z_cent  = _z_factory(df_ok, "centroid_hz")
    z_roll  = _z_factory(df_ok, "rolloff_hz")
    z_metl  = _z_factory(df_ok, "metallic_hint")

    # ---------------- scoring model (hat-specific) ----------------
    # We don’t do "if-else". We compute scores for every bucket and pick the max.
    def _scores(r):
        dur   = float(r["dur_ms"])
        d24   = float(r["decay24_ms"])
        hf    = float(r["hf_ratio"])
        air   = float(r["air_ratio"])
        low   = float(r["low_ratio"])
        flat  = float(r["flatness"])
        zcr   = float(r["zcr"])
        sharp = float(r["sharp"])
        cent  = float(r["centroid_hz"])
        roll  = float(r["rolloff_hz"])
        metl  = float(r["metallic_hint"])

        # gates (soft bonuses)
        gate_hf   = 0.5 if hf >= Q["hf_hi"] else 0.0
        gate_air  = 0.4 if air >= Q["air_hi"] else 0.0
        gate_tight= 0.6 if (d24 <= Q["d24_lo"] and dur <= Q["dur_lo"]) else 0.0
        gate_open = 0.6 if (d24 >= Q["d24_hi"] and dur >= Q["dur_hi"]) else 0.0
        gate_noisy= 0.6 if (flat >= Q["flat_hi"] and zcr >= _q(df_ok["zcr"], 75)) else 0.0

        sc = {}

        # 01 Closed Tight: short tail, sharp transient, HF present but controlled
        sc["01_CLOSED_TIGHT"] = (
            -1.6 * z_d24(d24) -
            1.0 * z_dur(dur) +
            1.2 * z_sharp(sharp) +
            0.6 * z_hf(hf) -
            0.5 * z_low(low)
        ) + gate_tight

        # 02 Closed Ticky: very crisp HF/air, sharp, still short-ish
        sc["02_CLOSED_TICKY"] = (
            1.4 * z_hf(hf) +
            1.0 * z_air(air) +
            0.9 * z_sharp(sharp) -
            1.0 * z_d24(d24) -
            0.4 * z_low(low)
        ) + gate_hf + gate_air

        # 03 Open Washy: long tail, lots of HF air, more noise-like than metallic
        sc["03_OPEN_WASHY"] = (
            1.6 * z_d24(d24) +
            1.1 * z_dur(dur) +
            0.9 * z_hf(hf) +
            0.6 * z_air(air) +
            0.5 * z_flat(flat)
        ) + gate_open

        # 04 Open Metallic: long-ish, more “peaky/metallic” (lower flatness, higher rolloff/centroid)
        sc["04_OPEN_METALLIC"] = (
            1.3 * z_d24(d24) +
            0.8 * z_dur(dur) +
            1.0 * z_cent(cent) +
            1.0 * z_roll(roll) -
            0.9 * z_flat(flat) +
            0.8 * z_metl(metl)
        ) + gate_open

        # 05 Shaker Hat: noise band focused, mid presence, high zcr, not too metallic
        sc["05_SHAKER_HAT"] = (
            1.2 * z_flat(flat) +
            1.0 * z_zcr(zcr) +
            0.6 * z_hf(hf) -
            0.6 * z_metl(metl) -
            0.4 * z_d24(d24)
        ) + (0.4 if (flat >= Q["flat_lo"] and hf >= Q["hf_lo"]) else 0.0)

        # 06 Noisy Texture: very flat/noisy, strong zcr, wideband
        sc["06_NOISY_TEXTURE"] = (
            1.7 * z_flat(flat) +
            1.3 * z_zcr(zcr) +
            0.5 * z_roll(roll) +
            0.4 * z_hf(hf)
        ) + gate_noisy

        # 07 Crashy Hat: very long + very wideband HF/air (almost cymbal)
        sc["07_CRASHY_HAT"] = (
            1.8 * z_d24(d24) +
            1.4 * z_dur(dur) +
            1.0 * z_roll(roll) +
            0.9 * z_air(air) +
            0.6 * z_hf(hf)
        ) + gate_open + gate_air

        # 08 Lofi Dirty: less HF/air, more low/mid junk, sometimes noisy
        sc["08_LOFI_DIRTY"] = (
            1.2 * z_low(low) -
            1.2 * z_hf(hf) -
            0.8 * z_air(air) +
            0.8 * z_flat(flat)
        ) + (0.4 if (low >= Q["low_hi"] and hf <= Q["hf_lo"]) else 0.0)

        # 09 Weird FX: catcher for unusual combos (high metallic + low hf, etc.)
        weird_bonus = 0.0
        if (metl >= _q(df_ok["metallic_hint"], 75) and hf <= Q["hf_lo"]):
            weird_bonus += 0.6
        if (flat <= Q["flat_lo"] and d24 >= Q["d24_hi"] and air <= _q(df_ok["air_ratio"], 25)):
            weird_bonus += 0.4
        sc["09_WEIRD_FX"] = (
            0.4 * z_metl(metl) +
            0.3 * z_cent(cent) +
            0.2 * z_flat(flat) +
            0.2 * z_d24(d24)
        ) + weird_bonus

        return sc

    # assign bucket + confidence margin
    assigned = []
    conf = []
    score_rows = []

    for r in df_ok.to_dict("records"):
        sc = _scores(r)
        items = sorted(sc.items(), key=lambda kv: kv[1], reverse=True)
        b1, s1 = items[0]
        b2, s2 = items[1]
        assigned.append(b1)
        conf.append(float(s1 - s2))
        score_rows.append(sc)

    df_ok["bucket"] = assigned
    df_ok["conf_margin"] = conf
    df_ok["_idx"] = np.arange(len(df_ok))

    score_df = pd.DataFrame(score_rows)
    score_df["_idx"] = df_ok["_idx"].values

    # ---------------- rebalance: avoid empty folders (only move borderline samples) ----------------
    def _rebalance_once(df_work):
        counts = df_work["bucket"].value_counts().to_dict()
        need = [b for b in buckets if counts.get(b, 0) < min_per_bucket]
        if not need:
            return df_work, False

        df_cand = df_work.sort_values("conf_margin", ascending=True).copy()
        moved = False

        for target in need:
            cur = counts.get(target, 0)
            add = max(0, min_per_bucket - cur)
            if add == 0:
                continue

            merged = df_cand.merge(score_df[["_idx", target]], on="_idx", how="left")
            merged = merged.rename(columns={target: "target_score"})
            merged = merged[merged["bucket"] != target].copy()

            merged = merged.sort_values(["target_score", "conf_margin"], ascending=[False, True])
            pick = merged.head(add)
            if len(pick) == 0:
                continue

            idxs = pick["_idx"].to_list()
            df_work.loc[df_work["_idx"].isin(idxs), "bucket"] = target
            moved = True

        return df_work, moved

    for _ in range(6):
        df_ok, changed = _rebalance_once(df_ok)
        if not changed:
            break

    # merge back
    df = df.merge(df_ok[["Path", "bucket", "conf_margin"]], on="Path", how="left")
    df["bucket"] = df["bucket"].fillna("09_WEIRD_FX")

    # ---------------- move/copy in-place ----------------
    if (mode.lower() in ["move", "copy"]) and (not dry_run):
        ok_rows = df[df["error"].eq("")]
        for r in tqdm(ok_rows.itertuples(index=False), desc=f"{mode.upper()} hats into folders", total=len(ok_rows)):
            src = r.Path
            bucket = r.bucket if isinstance(r.bucket, str) else "09_WEIRD_FX"
            dst_dir = bucket_paths.get(bucket, bucket_paths["09_WEIRD_FX"])
            dst = _mk_dest(dst_dir, src)

            if os.path.abspath(src) == os.path.abspath(dst):
                continue

            if mode.lower() == "copy":
                shutil.copy2(src, dst)
            else:
                shutil.move(src, dst)

    # ---------------- save manifest + summary ----------------
    out_csv = os.path.join(in_dir, "hat_manifest.csv")
    df.to_csv(out_csv, index=False)

    counts = df[df["error"].eq("")]["bucket"].value_counts().reindex(buckets).fillna(0).astype(int)
    df_summary = counts.reset_index()
    df_summary.columns = ["bucket", "count"]

    return df, df_summary

In [8]:
# ============================================================
#!#!#!#!#! RUNNING STATEMENTS #!#!#!#!#!
# ============================================================

audio_extensions = [".mp3"]  # add more if you want

df_hats, df_summary = _hat_0403_smartbucket_inplace_GET_df_manifest(
    in_dir=out_dir,
    audio_extensions=audio_extensions,
    mode="move",          # moves originals into created subfolders
    min_per_bucket=10,    # raise/lower depending on folder size
    dry_run=False
)

print(df_summary)
print(df_hats[["file_name","bucket","conf_margin","hf_ratio","air_ratio","low_ratio","flatness","decay24_ms","sharp","error"]].head(30))

Extracting hi-hat features: 0it [00:00, ?it/s]


KeyError: 'error'

In [ ]:
# END 
print("HATS - DONE")